# Final preprocessing of the unlabeled multiclass test dataset

This notebook preprocesses all images in the unlabeled multiclass test dataset.

For each image:

1. Crop the black borders of the dermatoscope.
2. Extract a central square and resize to 512 × 512.
3. Apply color constancy (Shades of Gray, p=6, with brightness normalization).
4. Remove hair using a cross-shaped black-hat filter, shape filtering and inpainting.
5. Segment the lesion using Otsu thresholding on the S channel, followed by morphological cleaning.

The preprocessing functions are defined in `preprocessing/`. This notebook sets the input and output paths, configures the parameters and processes the images.

**Input**
- Images: `test_multiclass/`
- The images are unlabeled; no ground-truth class is assigned.

**Output**
- Preprocessed images: `preprocessed_test_multiclass/test/unlabeled/`
- Manifest: `preprocessed_test_multiclass/manifest.csv`

For each image, the notebook saves:

- `<name>_preprocessed.png`: RGB image, 512 × 512, saved as a lossless PNG.
- `<name>_mask.png`: binary mask with values 0 and 255, `uint8`, 512 × 512.

The process is resumable: if both output files already exist, an image is not processed again unless `OVERWRITE = True`.

The same preprocessing parameters are used for the binary and multiclass datasets to ensure consistency.

**Important:** this notebook performs preprocessing and segmentation only. It does not assign classes, predict labels or train a model.

In [1]:
# %% Imports and configuration

%load_ext autoreload
%autoreload 2

import os
import sys
import time
from pathlib import Path

# The preprocessing package lives in the repository root
REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "preprocessing").is_dir()
)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(REPO), os.environ.get("PYTHONPATH", "")]
)

from preprocessing import (
    PreprocParams,
    run_dataset,
    summarize_manifest,
    update_manifest,
)

# ---------------- Configuration ----------------

ROOT = REPO
TEST_DIR = ROOT / "test_multiclass"
OUT_DIR = ROOT / "preprocessed_test_multiclass"

SEED = 42
OVERWRITE = False
N_JOBS = 4


# Preprocessing parameters: the SAME in Binary and Multiclass (defaults in preprocessing/params.py)
P = PreprocParams(
    dark_thr=35,                  # 1. borders: gray < dark_thr = black
    min_area_frac=0.08,           # 1. borders: if the useful rectangle is smaller than this fraction, no crop is done
    target_size=512,              # 2. central square + resize
    sog_p=6,                      # 3. Shades of Gray
    normalize_brightness=True,    # 3. True = the estimated light is mapped to gray sog_target; False = only the tint is corrected
    sog_target=195.0,
    hair=dict(kernel_size=5,      # 4. hair (teammate's method): black-hat kernel width > hair thickness
              thresh=15,          #    minimum black-hat value to count as hair
              kernel_shape="cross",
              channel="gray",
              blur_ksize=3,       #    smooths the black-hat map to join broken hairs
              filter_shape=True,  #    discards round blobs (dots, lesion borders)
              min_elongation=3.0,
              min_area=20,
              max_fill=0.35,
              dilate_iter=1,
              inpaint_radius=5,
              inpaint_method="telea"),
    hair_ref_side=256,            # 4. hair is detected at this size (where the values were tuned) and inpainted at 512
    seg_channel="S",              # 5. channel for Otsu: "S", "L", "a", "b", "Cr", "gray"
)

## Stratified selection (class x subtype, real proportions)

In [2]:

# %% Test image selection

TEST_DIR = Path(TEST_DIR)
OUT = Path(OUT_DIR)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

if not TEST_DIR.is_dir():
    raise FileNotFoundError(
        f"{TEST_DIR} does not exist. Check the test dataset path."
    )

# Find images directly inside test_binary, without class subfolders
test_images = sorted(
    f for f in TEST_DIR.iterdir()
    if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
)

if not test_images:
    raise RuntimeError(
        f"No image files found directly inside {TEST_DIR}"
    )

# The class label is organizational only; no predictions are assigned
jobs = [
    ("test", "unlabeled", f)
    for f in test_images
]

print("Input directory: ", TEST_DIR)
print("Output directory:", OUT)
print(f"\nTotal test images found: {len(jobs)}")

print("\nFirst 10 images:")
for f in test_images[:10]:
    print(" ", f.name)

Input directory:  C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\test_multiclass
Output directory: C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed_test_multiclass

Total test images found: 2121

First 10 images:
  xxx00001.jpg
  xxx00002.jpg
  xxx00003.jpg
  xxx00004.jpg
  xxx00005.jpg
  xxx00006.jpg
  xxx00007.jpg
  xxx00008.jpg
  xxx00009.jpg
  xxx00010.jpg


## Processing and saving (parallel and resumable)

In [3]:
print("Parameters used:\n")
print(P.table().to_string(index=False), "\n")

t0 = time.time()
rows = run_dataset(jobs, OUT, P, overwrite=OVERWRITE, n_jobs=N_JOBS)
print(f"\nTotal time: {time.time() - t0:.0f}s  ({(time.time() - t0) / max(len(jobs), 1):.2f} s/img wall clock)")

Parameters used:

                     step            parameter  value
               1. Borders             dark_thr     35
               1. Borders        min_area_frac   0.08
                2. Resize          target_size    512
3. Color (Shades of Gray)                sog_p      6
3. Color (Shades of Gray) normalize_brightness   True
3. Color (Shades of Gray)           sog_target  195.0
                  4. Hair     hair.kernel_size      5
                  4. Hair          hair.thresh     15
                  4. Hair    hair.kernel_shape  cross
                  4. Hair         hair.channel   gray
                  4. Hair      hair.blur_ksize      3
                  4. Hair    hair.filter_shape   True
                  4. Hair  hair.min_elongation    3.0
                  4. Hair        hair.min_area     20
                  4. Hair        hair.max_fill   0.35
                  4. Hair     hair.dilate_iter      1
                  4. Hair  hair.inpaint_radius      5
          

  0%|          | 0/2121 [00:00<?, ?it/s]


Total time: 167s  (0.08 s/img wall clock)


## Manifest and summary

In [4]:
manifest_path = OUT / "manifest.csv"
new = update_manifest(rows, manifest_path)       # keeps full data from previous runs
print("manifest:", manifest_path, f"({len(new)} rows)\n")
summarize_manifest(new)

manifest: C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed_test_multiclass\manifest.csv (2121 rows)

status
ok    2121 

                    n  seg_ok  area_frac_med  hair_frac_med
split class                                                
test  unlabeled  2121   0.335          0.189            0.0

Not cropped (fallback): 0
